### Data Reading

In [0]:
spark.sql("SHOW CATALOGS").display()

In [0]:
df = spark.table("workspace.default.big_mart_sales")

In [0]:
df.display()

### Data Reading JSON

In [0]:
df_json = (
    spark.read.format("json")
    .option("inferSchema", True)
    .option("header", True)
    .option('multiLine', False)
    .load("/Volumes/workspace/default/tutorial_volume/drivers.json")
)

In [0]:
df_json.display()


### Schema Definition

In [0]:
df.printSchema()


### DDL Schema

In [0]:
my_ddl_schema = """
                Item_Identifier STRING,
                Item_Weight DOUBLE,
                Item_Fat_Content STRING,
                Item_Visibility DOUBLE,
                Item_Type STRING,
                Item_MRP DOUBLE,
                Outlet_Identifier STRING,
                Outlet_Establishment_Year INT,
                Outlet_Size STRING,
                Outlet_Location_Type STRING,
                Outlet_Type STRING,
                Item_Outlet_Sales DOUBLE
"""

In [0]:
df = (
    spark.read.format("csv")
    .schema(my_ddl_schema)
    .option("header", True)
    .load("/Volumes/workspace/default/tutorial_volume/BigMart Sales.csv")
)

In [0]:
df.display()


### StructType() Schema

In [0]:
from pyspark.sql.types import *
from pyspark.sql.functions import *

In [0]:
my_struct_schema = StructType([
    StructField('Item_Identifier', StringType(), True),
    StructField('Item_Weight', DoubleType(), True),
    StructField('Item_Fat_Content', StringType(), True),
    StructField('Item_Visibility', DoubleType(), True),
    StructField('Item_Type', StringType(), True),
    StructField('Item_MRP', DoubleType(), True),
    StructField('Outlet_Identifier', StringType(), True),
    StructField('Outlet_Establishment_Year', IntegerType(), True),
    StructField('Outlet_Size', StringType(), True),
    StructField('Outlet_Location_Type', StringType(), True),
    StructField('Outlet_Type', StringType(), True),
    StructField('Item_Outlet_Sales', DoubleType(), True)
])

In [0]:
df = (
    spark.read.format("csv")
    .schema(my_struct_schema)
    .option("header", True)
    .load("/Volumes/workspace/default/tutorial_volume/BigMart Sales.csv")
)

In [0]:
df.display()

### SELECT

In [0]:
df_sel = df.select(col('Item_Identifier'), col('Item_Weight'), col('Item_Fat_Content'))
df_sel.display()

### ALIAS

In [0]:
df.select(col("Item_Identifier").alias("Item_ID")).display()

### FILTER

#### Scenario 1

In [0]:
df.filter(col('Item_Fat_Content') == 'Regular').display()

#### Scenario 2

In [0]:
df.filter((col('Item_Type') == 'Soft Drinks') & (col('Item_Weight') < 10)).display()


#### Scenario 3

In [0]:
df.filter((col('Outlet_Location_Type').isin('Tier 1', 'Tier 2')) & (col('Outlet_Size').isNull())).display()

### withColumnRenamed

In [0]:
df.withColumnRenamed('Item_Weight', 'Item_Wt').display()

### withColumn

#### Scenario 1

In [0]:
df = df.withColumn('flag', lit('new'))

In [0]:
df.display()

In [0]:
df.withColumn('multiply', col('Item_Weight') * col('Item_MRP')).display()

#### Scenario 2

In [0]:
df.withColumn('Item_Fat_Content', regexp_replace(col('Item_Fat_Content'), 'Regular', 'Reg')) \
    .withColumn('Item_Fat_Content', regexp_replace(col('Item_Fat_Content'), 'Low Fat', 'LF')).display()

### Type Casting

In [0]:
df = df.withColumn('Item_Weight', col('Item_Weight').cast(StringType()))

In [0]:
df.printSchema()

### Sort

#### Scenario 1

In [0]:
df.sort(col('Item_Weight').desc()).display()

#### Scenario 2

In [0]:
df.sort(col('Item_Visibility').asc()).display()

#### Scenario 3

In [0]:
df.sort(['Item_Weight', 'Item_Visibility'], ascending=[False, True]).display()

#### Scenario 4

In [0]:
df.sort(['Item_Weight', 'Item_Visibility'], ascending = [0, 1]).display()

### Limit

In [0]:
df.limit(10).display()

### Drop

#### Scenario 1

In [0]:
df.drop('Item_Visibility').display()

#### Scenario 2

In [0]:
df.drop('Item_Visibility', 'Item_Type').display()

### Drop_duplicates

In [0]:
df.drop_duplicates().display()

#### Scenario 2

In [0]:
df.drop_duplicates(subset=['Item_Type']).display()

In [0]:
df.distinct().display()

### Union

#### Preparing Dataframes

In [0]:
data1 = [('1', 'kad'),
         ('2', 'sid')]

schema1 = 'id STRING, name STRING'

df1 = spark.createDataFrame(data = data1, schema = schema1)

data2 = [('3', 'rahul'),
         ('4', 'jas')]

schema2 = 'id STRING, name STRING'

df2 = spark.createDataFrame(data = data2, schema = schema2)

In [0]:
df1.display()

In [0]:
df2.display()

### Union

In [0]:
df1.union(df2).display()

In [0]:
data1 = [('kad', '1'),
         ('sid', '2')]

schema1 = 'name STRING, id STRING'

df1 = spark.createDataFrame(data = data1, schema = schema1)

In [0]:
df1.union(df2).display()

### UnionByName

In [0]:
df1.unionByName(df2).display()

### String functions

#### Initcap

In [0]:
df.select(initcap('Item_Type')).display()

#### Upper

In [0]:
df.select(upper('Item_Type')).display()

#### Lower

In [0]:
df.select(lower('Item_Type').alias('Lower_Item_Type')).display()

### Date functions

#### Current_date

In [0]:
df = df.withColumn('Current date', current_date())
df.display()

### Date_add

In [0]:
df = df.withColumn('Future date', date_add('Current date', 7))

df.display()

#### Date_sub

In [0]:
df = df.withColumn('Previous date', date_sub('Current date', 7))

df.display()

### DateDiff

In [0]:
df = df.withColumn('date_diff', date_diff('Future Date', 'Current Date'))

df.display()

### Date_format

In [0]:
df = df.withColumn('Previous Date', date_format('Previous Date', 'dd-MM-yyyy'))

df.display()

### Handling nulls

#### Dropping Nulls

In [0]:
df.dropna('any').display()

In [0]:
df.dropna(subset=['Outlet_Size']).display()

#### Filling Nulls

In [0]:
df.fillna('Not Avialable').display()

In [0]:
df.fillna('Not Avaiable', subset=['Outlet_Size']).display()

### Split and Indexing

In [0]:
df.withColumn('Outlet_Type', split('Outlet_Type', ' ')).display()

#### Indexing

In [0]:
df.withColumn('Outlet_Type', split('Outlet_Type', ' ')[1]).display()

### Explode

In [0]:
df_exp = df.withColumn('Outlet_Type', split('Outlet_Type', ' '))

df_exp.display()

In [0]:
df_exp.withColumn('Outlet_Type', explode('Outlet_Type')).display()

### Array_contains

In [0]:
df_exp.withColumn('Type1_flag', array_contains('Outlet_Type', 'Type1')).display()

### Group_by

#### Scenario 1

In [0]:
df.groupBy('Item_Type').agg(sum('Item_MRP')).display()

#### Scenario 2

In [0]:
df.groupBy('Item_Type').avg(('Item_MRP')).display()

#### Scenario 3

In [0]:
df.groupBy(['Item_Type', 'Outlet_Size']).sum('Item_MRP').alias('Total_MRP').display()

#### Scenario 4

In [0]:
df.groupBy(['Item_Type', 'Outlet_Size']).agg(
    sum('Item_MRP').alias('Total_MRP'),
    avg('Item_MRP').alias('Avg_MRP')
).display()